In [2]:
from pymatgen.io.cif import CifParser   # 更完整地读取 cif 文件信息（包括键长等）
from rdkit import Chem
from rdkit.Chem import AllChem
from all_frags import Molecule
from all_frags import PROBE_FRAGMENTS, REF_FRAGMENTS
import numpy as np
from scipy.spatial.distance import cdist

In [3]:
cod_id_list = ["1100743"]
# for cod_id in cod_id_list:
#     pass
cod_id = cod_id_list[0]

supplier = Chem.SDMolSupplier("/home/clasper/2507/codes/1100743_test.sdf")
mol = supplier[0]

In [4]:
def SimplifyMol(mol):
    """Change all bonds to single and discharge/dearomatize all atoms.
    The molecule is modified in-place (no copy is made).
    from https://github.com/oddt/oddt/blob/master/oddt/toolkits/extras/rdkit/fixer.py
    """
    for b in mol.GetBonds():
        b.SetBondType(Chem.BondType.SINGLE)
        b.SetIsAromatic(False)
    for a in mol.GetAtoms():
        a.SetFormalCharge(0)
        a.SetIsAromatic(False)

    return mol


def get_heavy_indices_from_smarts(smarts):
    """
    Parses a SMARTS string and returns a list of indices for atoms that are
    explicitly defined as heavy atoms (i.e., not '*' or '[H]').
    """
    mol = Chem.MolFromSmarts(smarts)
    if not mol:
        print(f"Warning: Could not parse SMARTS string: {smarts}", flush=True)
        return []
    
    heavy_atom_indices = []
    for atom in mol.GetAtoms():
        # Get the SMARTS representation of the single atom
        atom_smarts = atom.GetSmarts()
        # We consider it a core heavy atom if it's not a generic wildcard or explicit hydrogen
        if atom_smarts not in ['*', '[H]', '[#1]']:
            heavy_atom_indices.append(atom.GetIdx())

    return heavy_atom_indices


def construct_template(smiles):
    template = Chem.MolFromSmiles(smiles)
    template = Chem.AddHs(template)
    AllChem.EmbedMolecule(template)
    order = []
    heavy_count = 0
    for a in template.GetAtoms():
        if a.GetAtomicNum() == 1:
            continue
        heavy_count += 1
        order.append(a.GetIdx())
        for n in a.GetNeighbors():
            if n.GetAtomicNum() == 1:
                order.append(n.GetIdx())
    template = Chem.RenumberAtoms(template, order)

    return template


molecule1 = REF_FRAGMENTS['frag_14']
molecule2 = REF_FRAGMENTS['frag_46']

smarts_1 = molecule1.smarts.replace("=", "")    # 尝试？？？
smarts_2 = molecule2.smarts.replace("=", "")
smiles_1 = molecule1.smiles
smiles_2 = molecule2.smiles
heavy_atom_1 = [atom.GetIdx() for atom in Chem.MolFromSmiles(smiles_1).GetAtoms() if atom.GetSymbol() != "H"]
heavy_atom_2 = [atom.GetIdx() for atom in Chem.MolFromSmiles(smiles_2).GetAtoms() if atom.GetSymbol() != "H"]
num_heavy_1 = len(heavy_atom_1)
num_heavy_2 = len(heavy_atom_2)
heavy_indices_1 = get_heavy_indices_from_smarts(smarts_1)
heavy_indices_1 = get_heavy_indices_from_smarts(smarts_1)
heavy_indices_2 = get_heavy_indices_from_smarts(smarts_2)
heavy_indices_2 = get_heavy_indices_from_smarts(smarts_2)

print(f"Fragment 1: {smarts_1} -> {smiles_1}", flush=True)
print(f"Fragment 2: {smarts_2} -> {smiles_2}", flush=True)
print(f"Heavy atoms: {num_heavy_1} and {num_heavy_2}", flush=True)
print("\nConstructing templates...", flush=True)
template_1 = construct_template(smiles_1)
template_2 = construct_template(smiles_2)
num_1 = template_1.GetNumAtoms()
num_2 = template_2.GetNumAtoms()
print(f"Template atoms: {num_1} and {num_2}", flush=True)
substruct1 = Chem.MolFromSmarts(smarts_1)
# substruct1 = Chem.MolFromSmarts("c1([*])c([*])c([*])c([*])c([*])c1([*])")
substruct2 = Chem.MolFromSmarts(smarts_2)
# substruct2 = Chem.MolFromSmarts("C1([*])([*])C([*])([*])N([*])C([*])([*])C([*])([*])C1([*])")
print("fragments: ", Chem.MolToSmarts(substruct1), Chem.MolToSmarts(substruct2))

Fragment 1: C([*])([*])([*])O([*]) -> C[OH]
Fragment 2: c1([*])c([*])c([*])c([*])c([*])c1([*]) -> c1ccccc1
Heavy atoms: 2 and 6

Constructing templates...
Template atoms: 6 and 12
fragments:  C(*)(*)(*)O* c1(*)c(*)c(*)c(*)c(*)c1*


In [5]:
# 优化 mol 结构（去除所有氢，芳香性判断，加上通配符）
rw_mol = Chem.RWMol(mol)
bonds_to_remove = []
i, j = 0, 0
for bond in rw_mol.GetBonds():
    atom1 = rw_mol.GetAtomWithIdx(bond.GetBeginAtomIdx())
    atom2 = rw_mol.GetAtomWithIdx(bond.GetEndAtomIdx())
    if atom1.GetAtomicNum() == 1 and atom2.GetAtomicNum() == 1:
        bonds_to_remove.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
if bonds_to_remove:
    for begin_idx, end_idx in bonds_to_remove:
        rw_mol.RemoveBond(begin_idx, end_idx)
        i += 1
h_atoms = [atom.GetIdx() for atom in rw_mol.GetAtoms() if atom.GetAtomicNum() == 1]
for idx in sorted(h_atoms, reverse=True):
    rw_mol.RemoveAtom(idx)
    j += 1
print(f"Removed {i} H-H and {j} Hs. ")
mol = rw_mol.GetMol()


tolerance = 0.05
sssr = Chem.GetSymmSSSR(mol)
if sssr:
    i, j = 0, 0
    for ring in sssr:
        coords = []
        conf = mol.GetConformer()
        for atom_idx in ring:
            position = conf.GetAtomPosition(atom_idx)
            # print(f"position: ({position.x},{position.y},{position.z})")
            coords.append(position)
        coords = np.array(coords)
        centroid = np.mean(coords, axis=0)
        coords_centered = coords - centroid
        # SVD 拟合平面
        _, _, vh = np.linalg.svd(coords_centered)
        normal = vh[2, :]
        # 计算距离
        distances = np.abs(np.dot(coords_centered, normal))
        max_deviation = np.max(distances)
        rms_deviation = np.sqrt(np.mean(distances**2))
        if max_deviation < tolerance:
            # print(f"Ring {[atom_idx for atom_idx in ring]} is aromatic. ")
            for atom_idx in ring:
                atom = mol.GetAtomWithIdx(atom_idx)
                atom.SetIsAromatic(True)
            for bond in mol.GetBonds():
                begin_idx = bond.GetBeginAtomIdx()
                end_idx = bond.GetEndAtomIdx()
                if begin_idx in ring and end_idx in ring:
                    bond.SetIsAromatic(True)
                    bond.SetBondType(Chem.BondType.AROMATIC)
            i += 1
        else:
            # print(f"Ring {[atom_idx for atom_idx in ring]} is not aromatic. ")
            j += 1
    print(f"Found {i} rings aromatic, while {j} not. ")

sanitize_ops = (
    Chem.SanitizeFlags.SANITIZE_FINDRADICALS |
    Chem.SanitizeFlags.SANITIZE_SETAROMATICITY |
    Chem.SanitizeFlags.SANITIZE_SETCONJUGATION |
    Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION |
    Chem.SanitizeFlags.SANITIZE_CLEANUP 
    # Chem.SanitizeFlags.SANITIZE_CHECKVALENCE 
    )
Chem.SanitizeMol(mol, sanitizeOps=sanitize_ops)
mol = Chem.AddHs(mol, addCoords=True)

Removed 0 H-H and 0 Hs. 
Found 96 rings aromatic, while 0 not. 


In [6]:
# 用于辅助后期完整分子的判断
fragments = Chem.GetMolFrags(mol, asMols=False)
fragment_num = {}
for fragment in fragments:
    num = len(fragment)
    if num not in fragment_num:
        fragment_num[num] = 0
    fragment_num[num] += 1
# 晶胞为相同分子的重复，则重复最多次的应是完整分子（其余为晶胞边缘处的片段）
valid_molecule_num = max(fragment_num.values())
valid_sizes = [size for size, num in fragment_num.items() if num == valid_molecule_num]
# print(valid_sizes)
valid_molecules = []
atom_frag_dict = {}
for frag_id, fragment in enumerate(fragments):
    if len(fragment) in valid_sizes:
        valid_molecules.append(frag_id)
    for atom_id in fragment:
        atom_frag_dict[atom_id] = frag_id
# print(atom_frag_dict)
print(valid_molecules)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]


In [7]:
# print("Before adding Hs:", mol.GetNumAtoms())
# Chem.AddHs(mol, addCoords=True)
# print("After adding Hs:", mol.GetNumAtoms())

# if mol:
#     i = 0
#     conf = mol.GetConformer()
#     for i in range(mol.GetNumAtoms()):
#         print(mol.GetAtomWithIdx(i).GetSymbol(), np.array(conf.GetAtomPosition(i)))

# substruct1 = Chem.MolFromSmarts("N([*])([*])C([*])O")
# substruct2 = Chem.MolFromSmarts("c1([*])c([*])c([*])c([*])c([*])c1([*])")
def search(mol, substruct1, substruct2):
    match1s = mol.GetSubstructMatches(substruct1)
    match2s = mol.GetSubstructMatches(substruct2)
    # print(type(match1s))
    # print(len(match1s), len(match2s))

    from scipy.spatial.distance import cdist


    hits = []
    # 获取第一个子结构的所有重原子索引
    heavy_indices_1s = {}
    for match1 in match1s:
        if not all([atom_frag_dict[i] == atom_frag_dict[i + 1] for i in range(len(match1) - 1)]):
            print(f"fragment {match1} on different molecule! ")
            continue
        if not atom_frag_dict[match1[0]] in valid_molecules:
            print("match1 not of molecule, continue...")
            continue
        heavy_indices_1 = set()
        for idx in match1:
            atom = mol.GetAtomWithIdx(idx)
            if atom.GetAtomicNum() != 1:
                heavy_indices_1.add(idx)
        if heavy_indices_1:
            heavy_indices_1s[frozenset(heavy_indices_1)] = match1
    print(len(heavy_indices_1s))
    # 获取第二个子结构的所有重原子索引
    heavy_indices_2s = {}
    for match2 in match2s:
        if not all([atom_frag_dict[i] == atom_frag_dict[i + 1] for i in range(len(match2) - 1)]):
            print(f"fragment {match2} on different molecule! ")
            continue
        if not atom_frag_dict[match2[0]] in valid_molecules:
            print("match2 not of molecule, continue...")
            continue
        heavy_indices_2 = set()
        for idx in match2:
            atom = mol.GetAtomWithIdx(idx)
            if atom.GetAtomicNum() != 1:
                heavy_indices_2.add(idx)
        if heavy_indices_2:
            heavy_indices_2s[frozenset(heavy_indices_2)] = match2
    print(len(heavy_indices_2s))
    conf = mol.GetConformer()
    i = 0
    for heavy_indices_1, match1 in heavy_indices_1s.items():
        coords_1 = np.array([conf.GetAtomPosition(i) for i in heavy_indices_1])
        for heavy_indices_2, match2 in heavy_indices_2s.items():
            if atom_frag_dict[match1[0]] != atom_frag_dict[match2[0]]:
                coords_2 = np.array([conf.GetAtomPosition(j) for j in heavy_indices_2])
                distances = cdist(coords_1, coords_2, metric='euclidean')
                if np.min(distances) < 5.0:
                    hit = (match1, match2)
                    hits.append(hit)
            else:
                # print(f"match1 and match2 are on the same molecule, pass...")
                i += 1
    print(f"Got {len(hits)} hits! Skipped {i} groups as they are of the same molecule. ")

    # 为提高效率，每一个晶胞只返回一个结果
    return hits[0]


hit = search(mol, substruct1, substruct2)

64
64
Got 132 hits! Skipped 0 groups as they are of the same molecule. 


In [9]:
def construct_frag(mol, indices):
    """
    Constrcts the fragment mol from the total mol according to the indices. 
    Returns the fragment mol -> Mol. 
    """
    idx_map = {}

    # 添加原子
    frag = Chem.EditableMol(Chem.Mol())
    for new_idx, old_idx in enumerate(indices):
        atom = mol.GetAtomWithIdx(old_idx)
        frag.AddAtom(atom)
        idx_map[old_idx] = new_idx
    frag = frag.GetMol()
    
    # 添加坐标
    old_conf = mol.GetConformer()
    new_conf = Chem.Conformer(len(indices))
    for new_idx, old_idx in enumerate(indices):
        position = old_conf.GetAtomPosition(old_idx)
        new_conf.SetAtomPosition(new_idx, position)
    frag.AddConformer(new_conf)

    # 添加键
    frag = Chem.EditableMol(frag)
    for old_idx in indices:
        atom = mol.GetAtomWithIdx(old_idx)
        for neighbor in atom.GetNeighbors():
            neigh_id = neighbor.GetIdx()
            if neigh_id in indices and old_idx < neigh_id:
                bond = mol.GetBondBetweenAtoms(old_idx, neigh_id)
                frag.AddBond(idx_map[old_idx], idx_map[neigh_id], bond.GetBondType())
    
    return frag.GetMol()


def construct_frag_pair(mol, indices1, indices2):
    """
    Constructs the fragment pair within a cell. 
    Return frag_pair -> (Mol, Mol)
    """
    molFrag1 = construct_frag(mol, indices1)
    molFrag2 = construct_frag(mol, indices2)
    return (molFrag1, molFrag2)


mol_pairs = []


# 为提高效率，一个晶胞中只取一个 hit
match1, match2 = hit
mol_pair = construct_frag_pair(mol, match1, match2)
mol_pairs.append(mol_pair)


# molFrag1 = mol_pair[0]
# conf = molFrag1.GetConformer()
# for i in range(molFrag1.GetNumAtoms()):
#     print((conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z))

In [11]:
import copy

def correct_H(mol, template, heavy_indices):
    """
    modify the hydrogen atom coordinates in the molecule based on a template
    """
    copy_mol = copy.deepcopy(mol)
    for i in range(copy_mol.GetNumAtoms()):
        if i not in heavy_indices:
            copy_mol.GetAtomWithIdx(i).SetAtomicNum(1)

    # 检查并移除H-H键
    rw_mol = Chem.RWMol(copy_mol)
    bonds_to_remove = []
    for bond in rw_mol.GetBonds():
        atom1 = rw_mol.GetAtomWithIdx(bond.GetBeginAtomIdx())
        atom2 = rw_mol.GetAtomWithIdx(bond.GetEndAtomIdx())
        if atom1.GetAtomicNum() == 1 and atom2.GetAtomicNum() == 1:
            bonds_to_remove.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
    
    if bonds_to_remove:
        for begin_idx, end_idx in bonds_to_remove:
            rw_mol.RemoveBond(begin_idx, end_idx)
    
    copy_mol = rw_mol.GetMol()

    simplified_mol = SimplifyMol(copy.deepcopy(copy_mol))
    simplified_template = SimplifyMol(copy.deepcopy(template))

    match_forward = simplified_template.GetSubstructMatch(simplified_mol)
    match_reverse = simplified_mol.GetSubstructMatch(simplified_template)

    if match_forward and match_reverse:
        template_to_mol_map = {template_idx: mol_idx for mol_idx, template_idx in enumerate(match_reverse)}

        rw_copy_mol = Chem.RWMol(copy_mol)
        for template_idx, mol_idx in template_to_mol_map.items():
            template_atom = template.GetAtomWithIdx(template_idx)
            mol_atom = rw_copy_mol.GetAtomWithIdx(mol_idx)
            
            if template_atom.GetAtomicNum() != 1:  
                # 设置电荷
                mol_atom.SetFormalCharge(template_atom.GetFormalCharge())
                
                # 计算该重原子在template中连接的氢原子数
                h_count = 0
                for neighbor in template_atom.GetNeighbors():
                    if neighbor.GetAtomicNum() == 1:  
                        h_count += 1
                
                # 设置显式氢原子数
                mol_atom.SetNumExplicitHs(h_count)
                # print(f"set atom {mol_idx} of mol(template {template_idx}): "
                #       f"charge={template_atom.GetFormalCharge()}, H count={h_count}")

        # 设置键类型
        for template_bond in template.GetBonds():
            template_begin_idx = template_bond.GetBeginAtomIdx()
            template_end_idx = template_bond.GetEndAtomIdx()
            
            mol_begin_idx = template_to_mol_map.get(template_begin_idx)
            mol_end_idx = template_to_mol_map.get(template_end_idx)
            
            if mol_begin_idx is not None and mol_end_idx is not None:
                mol_bond = rw_copy_mol.GetBondBetweenAtoms(mol_begin_idx, mol_end_idx)
                if mol_bond is not None:
                    template_bond_type = template_bond.GetBondType()
                    mol_bond.SetBondType(template_bond_type)
                    # print(f"set bond {mol_begin_idx}-{mol_end_idx}: {template_bond_type}")

        copy_mol = rw_copy_mol.GetMol()

        H_cor_dict = {}
        for t_bond in template.GetBonds():
            t_atom1 = t_bond.GetBeginAtom()
            t_atom2 = t_bond.GetEndAtom()

            is_polar_h = False
            if t_atom1.GetSymbol() == 'H' and t_atom2.GetSymbol() != 'H':
                t_h_idx, t_heavy_idx = t_atom1.GetIdx(), t_atom2.GetIdx()
                is_polar_h = True
            elif t_atom2.GetSymbol() == 'H' and t_atom1.GetSymbol() != 'H':
                t_h_idx, t_heavy_idx = t_atom2.GetIdx(), t_atom1.GetIdx()
                is_polar_h = True

            if is_polar_h:
                target_length = get_bond_length(template, t_heavy_idx, t_h_idx)
                
                mol_heavy_idx = template_to_mol_map.get(t_heavy_idx)
                mol_h_idx = template_to_mol_map.get(t_h_idx)

                if mol_heavy_idx is None or mol_h_idx is None:
                    continue

                heavy_pos = np.array(copy_mol.GetConformer().GetAtomPosition(mol_heavy_idx))
                heavy_coord_key = tuple(heavy_pos.tolist())

                h_pos = np.array(copy_mol.GetConformer().GetAtomPosition(mol_h_idx))
                bond_vector = h_pos - heavy_pos
                current_length = np.linalg.norm(bond_vector)
                
                if current_length > 1e-6:  
                    unit_vector = bond_vector / current_length
                    new_h_pos = heavy_pos + unit_vector * target_length
                    
                    if heavy_coord_key not in H_cor_dict:
                        H_cor_dict[heavy_coord_key] = []
                    H_cor_dict[heavy_coord_key].append(new_h_pos)

        mol_rmHs = Chem.RemoveHs(copy.deepcopy(copy_mol))
        mol_addHs = Chem.AddHs(mol_rmHs, addCoords=True)
        # for atom in mol_addHs.GetAtoms():
        #     print(f"Atom: {atom.GetIdx()}, Symbol: {atom.GetSymbol()}, Position: {mol_addHs.GetConformer().GetAtomPosition(atom.GetIdx())}", flush=True)

        for heavy_coord, correct_H_coords in H_cor_dict.items():
            heavy_coord = np.array(heavy_coord)
            
            # 根据坐标找到重原子的索引
            heavy_idx = None
            for i in range(mol_addHs.GetNumAtoms()):
                if np.allclose(mol_addHs.GetConformer().GetAtomPosition(i), heavy_coord, atol=1e-3):
                    heavy_idx = i
                    break
            
            if heavy_idx is None:
                continue
                
            # 找到该重原子的所有氢邻居
            H_indices = [atom.GetIdx() for atom in mol_addHs.GetAtomWithIdx(heavy_idx).GetNeighbors() 
                        if atom.GetSymbol() == "H"]
            
            # 将校正后的坐标应用到氢原子上
            for i, h_idx in enumerate(H_indices):
                mol_addHs.GetConformer().SetAtomPosition(h_idx, correct_H_coords[i])

        return (True, mol_addHs)

    else:
        print("Debug: mol and template do not match, skipping hydrogen correction", flush=True)
        return (False, copy_mol)
    

def get_bond_length(mol, atom1_idx, atom2_idx):
    """
    calculate the bond length between two atoms in a molecule
    """
    conf = mol.GetConformer()
    pos1 = np.array(conf.GetAtomPosition(atom1_idx))
    pos2 = np.array(conf.GetAtomPosition(atom2_idx))
    return np.linalg.norm(pos2 - pos1)


correctH_mol_pairs = []
correctH_mol_names = []
not_corrected_pairs = []
not_corrected_names = []
not_corrected_pairs_Hreplace = []

for i, (frag1_mol, frag2_mol) in enumerate(mol_pairs):
    if i % 50 == 0:
        print(f"Processing molecule pair {i+1}/{len(mol_pairs)}", flush=True)
    if_correct_1, corrected_frag1 = correct_H(frag1_mol, template_1, heavy_indices_1)
    if_correct_2, corrected_frag2 = correct_H(frag2_mol, template_2, heavy_indices_2)
    if not if_correct_1 or not if_correct_2:
        print("Failed to correct. ")
        combined_mol_not_corrected = Chem.CombineMols(frag1_mol, frag2_mol)
        not_corrected_pairs.append(combined_mol_not_corrected)
        not_corrected_names.append(cod_id)  
        combined_mol_not_corrected_Hreplace = Chem.CombineMols(corrected_frag1, corrected_frag2)
        not_corrected_pairs_Hreplace.append(combined_mol_not_corrected_Hreplace)
        continue
    print("Corrected. ")
    combined_mol = Chem.CombineMols(corrected_frag1, corrected_frag2)
    correctH_mol_pairs.append(combined_mol)
    correctH_mol_names.append(cod_id)

Processing molecule pair 1/1
Corrected. 


In [12]:
def align_sidemol(ref_central_mol, qry_mol, mol_idx):
    qry_mols = Chem.GetMolFrags(qry_mol, asMols=True)
    # align central molecule of qry to ref
    atomMap = [(i,i) for i in range(num_heavy_1)]
    rmsd, transform = Chem.rdMolAlign.GetAlignmentTransform(qry_mols[0], ref_central_mol, atomMap=atomMap)
    rotation = transform[:3, :3]
    translation = transform[:3, 3]
    if rmsd > 0.5:
        print(f"{mol_idx+1} align error: {rmsd}", flush=True)
    # for qry_mol in qry_mols:
    #     # apply the transformation to the molecule coordinates
    conf = qry_mol.GetConformer()
    for atom_idx in range(qry_mol.GetNumAtoms()):
        pos = np.array(conf.GetAtomPosition(atom_idx))
        new_pos = np.dot(rotation, pos) + translation
        conf.SetAtomPosition(atom_idx, new_pos)
    return qry_mol


print("\nAligning molecules...", flush=True)
ref_mol = correctH_mol_pairs[0]
ref_mols = Chem.GetMolFrags(ref_mol, asMols=True)
ref_central_mol, ref_side_mol = ref_mols[0], ref_mols[1]
aligned_mols = []
aligned_names = []

for i, qry_mol in enumerate(correctH_mol_pairs):
    if i % 50 == 0:
        print(f"Aligning molecule {i+1}/{len(correctH_mol_pairs)}", flush=True)
    if i == 0:
        aligned_mols.append(qry_mol)
        aligned_names.append(correctH_mol_names[i])
        continue
    qry_mol = align_sidemol(ref_central_mol, qry_mol, i)
    aligned_mols.append(qry_mol)
    aligned_names.append(correctH_mol_names[i])

print("Alignment completed.", flush=True)


Aligning molecules...
Aligning molecule 1/1
Alignment completed.


In [13]:
atomMap = [[(i,i) for i in range(num_heavy_2)]]
# delete similar molecules
print("\nRemoving redundant molecules...", flush=True)
nonredun_mols = []
nonredun_names = []

for i, qry_mol in enumerate(aligned_mols):
    if i % 50 == 0:
        print(f"Checking redundancy for molecule {i+1}/{len(aligned_mols)}", flush=True)
    redun_flag = False
    for j, ref_mol in enumerate(aligned_mols):
        if j <= i:
            continue
        qry_side_mol = Chem.GetMolFrags(qry_mol, asMols=True)[1]
        ref_side_mol = Chem.GetMolFrags(ref_mol, asMols=True)[1]
        rmsd = Chem.rdMolAlign.CalcRMS(ref_side_mol, qry_side_mol, map=atomMap)
        if rmsd < 0.00001:
            print(f"Too similar, delete {i+1} and {j+1}, RMSD={rmsd:.6f}", flush=True)
            redun_flag = True
            break
    if not redun_flag:
        nonredun_mols.append(qry_mol)
        nonredun_names.append(aligned_names[i])

print(f"Redundancy removal completed. Saving {len(nonredun_mols)} molecules", flush=True)


Removing redundant molecules...
Checking redundancy for molecule 1/1
Redundancy removal completed. Saving 1 molecules


In [14]:
# save the aligned molecule with names
output_file = "/home/clasper/2507/codes/test.sdf"
with Chem.SDWriter(output_file) as writer:
    for mol, name in zip(nonredun_mols, nonredun_names):
        mol.SetProp("_Name", name)  
        writer.write(mol)
print(f"Total molecules saved: {len(nonredun_mols)}", flush=True)

Total molecules saved: 1


In [ ]:
# save the not corrected pairs with names
not_corrected_file = "/home/clasper/2507/codes/not_corrected_test.sdf"
with Chem.SDWriter(str(not_corrected_file)) as writer:
    for mol, name in zip(not_corrected_pairs, not_corrected_names):
        try:
            Chem.SanitizeMol(mol)
            mol.SetProp("_Name", name)  
            writer.write(mol)
        except Exception as e:
            print(f"Error sanitizing molecule {name}: {e}", flush=True)
            continue
print(f"Not corrected pairs saved to: {not_corrected_file}", flush=True)

print(f"\nWorkflow completed! Final output: {output_file}, Not corrected pairs: {not_corrected_file}", flush=True)
print("=" * 50, flush=True)

In [ ]:
# 测试以追加形式写入新分子
# with Chem.SDWriter(open("/home/clasper/2507/codes/test.sdf", 'a')) as writer:
#     for mol in supplier:
#         if mol:
#             mol.SetProp("cod_id", "1100743")
#             writer.write(mol)

# # 测试读取
# test_supplier = Chem.SDMolSupplier("test.sdf")
# print(len(test_supplier))

In [ ]:
from pymatgen.io.cif import CifParser


file_path = f"/mnt/e/Users/clasper/tmp/cif/{cod_id[0]}/{cod_id[1:3]}/{cod_id[3:5]}/{cod_id}.cif"
parser = CifParser(file_path)
structure = parser.parse_structures(primitive=False)[0]
formula = structure.formula
cif_data = parser.as_dict()[cod_id]
mol_per_cell = cif_data.get('_cell_formula_units_Z')
# if mol_per_cell:
#     if int(mol_per_cell) < 2:
#         structure *= (3, 3, 3)
#     else:
#         structure *= (2, 2, 2)
# else:
#     structure *= (3, 3, 3)
print(len(structure))
if len(structure) > 99999: 
    print("Warning: structure too large! The .pdb file may be wrong. ")

150


/home/clasper/micromamba/envs/csd/lib/python3.11/site-packages/pymatgen/io/cif.py:1313: UserWarning: Incorrect stoichiometry:
  CIF={'C': 32.0, 'H': 24.0, 'N': 4.0, 'Ni': 1.0, 'O': 5.0, 'P': 1.0, 'S': 4.0}
  PMG={'Ni': 2.0, 'P': 2.0, 'H': 56.0, 'C': 64.0, 'S': 8.0, 'N': 8.0, 'O': 10.0}
  ratios={'P': 2.0, 'C': 2.0, 'S': 2.0, 'N': 2.0, 'Ni': 2.0, 'H': 2.3333333333333335, 'O': 2.0}
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):


In [67]:
atom_dict = {}
for i, site in enumerate(structure):
    dict = site.as_dict()
    # print(dict)
    species = dict['species'][0]
    # print(dict)
    # atom_dict[i] = [site.coords, dict['label'], atom_info['element']]
    atom_dict[i] = {"position": site.coords, "label": dict['label'], 'symbol': species['element'], 'occu': species['occu']}
    if 'oxidation_state' in species:
        atom_dict[i]['charge'] = species['oxidation_state']
    # break
    # if species['element'] == "H":
    #     print(i, atom_dict[i])
# print(atom_dict)

cryst_dict = {
    "a": 0.0, 
    "b": 0.0, 
    "c": 0.0, 
    "alpha": 90.0, 
    "beta": 90.0, 
    "gamma": 90.0, 
    "sGroup": "", 
    "z": 1
}
if '_cell_length_a' in cif_data:
    cryst_dict['a'] = float(cif_data['_cell_length_a'].replace("(", "").replace(")", ""))
if '_cell_length_b' in cif_data:
    cryst_dict['b'] = float(cif_data['_cell_length_b'].replace("(", "").replace(")", ""))
if '_cell_length_c' in cif_data:
    cryst_dict['c'] = float(cif_data['_cell_length_c'].replace("(", "").replace(")", ""))
if '_cell_angle_alpha' in cif_data:
    cryst_dict['alpha'] = float(cif_data['_cell_angle_alpha'].replace("(", "").replace(")", ""))
if '_cell_angle_beta' in cif_data:
    cryst_dict['beta'] = float(cif_data['_cell_angle_beta'].replace("(", "").replace(")", ""))
if '_cell_angle_gamma' in cif_data:
    cryst_dict['gamma'] = float(cif_data['_cell_angle_gamma'].replace("(", "").replace(")", ""))
if '_symmetry_space_group_name_H-M' in cif_data:
    cryst_dict['sGroup'] = cif_data['_symmetry_space_group_name_H-M']
if '_cell_formula_units_Z' in cif_data:
    cryst_dict['z'] = int(cif_data['_cell_formula_units_Z'])
# print(type(cryst_dict["a"]))

# 获取成键原子信息
bond_dict = {}
try:
    label1 = cif_data['_geom_bond_atom_site_label_1']
    label2 = cif_data['_geom_bond_atom_site_label_2']
    labels = list(zip(label1, label2))
    # print(labels)
    # 获取所有原子的相邻成键原子（但在 CIF 文件中通常不包括 H ）
    bond_label_dict = {}
    for label1, label2 in labels:
        if label1 not in bond_label_dict:
            bond_label_dict[label1] = []
        bond_label_dict[label1].append(label2)
        # if label2 not in bond_label_dict:
        #     bond_label_dict[label2] = []
        # bond_label_dict[label2].append(label1)
    # print(bond_label_dict)
    # 由于成键两端标签和两端索引是一对多的关系，所以必须把标签字典转换成索引字典 (bond_dict)
    for label1, label2s in bond_label_dict.items():
        bond_indexes_1 = [index for index, info in atom_dict.items() if label1 == info['label']]
        for bond_index_1 in bond_indexes_1:
            bond_dict[bond_index_1] = set()
            
            # 建立嵌套子对象，存储候选 bond_indexes_2 的各个索引及坐标
            site_1 = structure[bond_index_1] # 成键原子 1 的 site, 用于计算距离，所用方法为: site_1.distance(site_2)
            for label2 in label2s:  # 获得候选成键原子的索引集合
                candidates_index_2 = [index for index, info in atom_dict.items() if label2 == info['label']]
                distances = []
                for candidate_index_2 in candidates_index_2:
                    site_2 = structure[candidate_index_2]
                    distance = site_1.distance(site_2)
                    distances.append(distance)
                # if min(distances) < 2:  # 添加距离判断，如果过大则表示两个原子不在同一分子上（位于晶胞的对侧），也就不可能成键
                #     print(min(distances))
                #     bond_index_2 = candidates_index_2[distances.index(min(distances))]  # 取出距离最近的索引作为成键原子
                #     bond_dict[bond_index_1].add(bond_index_2)
                # print(min(distances))
                bond_index_2 = candidates_index_2[distances.index(min(distances))]  # 取出距离最近的索引作为成键原子
                bond_dict[bond_index_1].add(bond_index_2)
    print(f"{cod_id}.cif: bond information extracted! ")
except KeyError:
    print(f"{cod_id}.cif has no bond information, searching for jmol version...")

1100743.cif: bond information extracted! 


In [68]:
# atom_line 初始化内容
hetatm_col = "HETATM"   # 类型
serial_col = 1  # 原子序号，最多 99999 [required]
name_col = ""  # 原子名称 [required]
altLoc_col = "" # 可替位置标记符
resName_col = "UNK"    # 残基名称
chainID_col = ""    # 链标识符
resSeq_col = 1  # 残基序列号
iCode_col = ""  # 残基插入码
x_col = 0.0 # [required]
y_col = 0.0 # [required]
z_col = 0.0 # [required]
occu_col = 1.0
temp_col = 5.0  # 温度因子
segID_col = ""  # 片段标识符（可选）
element_col = ""    # 元素符号  # [required]
charge_col = "" # 原子电荷（可选）  # [required]


with open(f"{cod_id}_test.pdb", "w", encoding="utf-8") as f:
    f.write(f"COMPND    {formula}\n")

    # 晶胞参数
    a_col = "{:>9.3f}".format(cryst_dict['a'])
    b_col = "{:>9.3f}".format(cryst_dict['b'])
    c_col = "{:>9.3f}".format(cryst_dict['c'])
    alpha_col = "{:>7.2f}".format(cryst_dict['alpha'])
    beta_col = "{:>7.2f}".format(cryst_dict['beta'])
    gamma_col = "{:>7.2f}".format(cryst_dict['gamma'])
    sGroup_col = "{:<11}".format(cryst_dict['sGroup'])
    z_col = "{:>4d}".format(cryst_dict['z'])
    cryst_line = f"CRYST1{a_col}{b_col}{c_col}{alpha_col}{beta_col}{gamma_col} {sGroup_col}{z_col}"
    f.write(f"{cryst_line}\n")
    
    for index, atom_info in atom_dict.items():
        serial_col = index + 1
        name_col = atom_info['label']
        x_col, y_col, z_col = atom_info['position']
        element_col = atom_info['symbol']
        occu_col = atom_info['occu']
        if 'charge' in atom_info:
            charge_col = atom_info['charge']
        
        # 格式化
        HETATM = "{:<6}".format(hetatm_col)
        serial = "{:>5d}".format(serial_col)
        name = "{:<4}".format(name_col)
        altLoc = "{:1}".format(altLoc_col)
        resName = "{:3}".format(resName_col)
        chainID = "{:1}".format(chainID_col)
        resSeq = "{:>4d}".format(resSeq_col)
        iCode = "{:1}".format(iCode_col)
        x = "{:>8.3f}".format(x_col)
        y = "{:>8.3f}".format(y_col)
        z = "{:>8.3f}".format(z_col)
        occu = "{:>6.2f}".format(occu_col)
        temp = "{:>6.2f}".format(temp_col)
        segID = "{:<4}".format(segID_col)
        element = "{:>2}".format(element_col)
        charge = "{:>2}".format(charge_col)
        atom_line = f"{HETATM}{serial} {name}{altLoc}{resName} {chainID}{resSeq}{iCode}   {x}{y}{z}{occu}{temp}      {segID}{element}{charge}"

        f.write(f"{atom_line}\n")
    
    # for i, js in bond_dict.items():
    #     for j in js:
    #         conect_line = "CONECT{0:>5}{1:>5}".format(i + 1, j + 1)
    #         f.write(f"{conect_line}\n")
    f.write("END\n")